# Oct3V1 Naming Style (best-effort spec)

This document explains how session folders under `Oct3V1/YYYY_MM_DD/` are named.  
The scheme preserves historical names; it’s pragmatic and imperfect.

---

## Directory layout

Oct3V1/
├─ YYYY_MM_DD/ # experiment day (calendar date of the recording)
│ ├─ <SessionName>/ # a single recording session
│ └─ <SessionName>_HH_MM/ # same session base, different start time (prevents overwrite)
└─ README_Naming.md # this file


- `YYYY_MM_DD/` = the **experiment day folder** (when the recording happened).
- `<SessionName>` = the actual session identifier (rules below).

---

## Session name pattern


<animalName>[ _<tag> ... ][ _p<partnerName> ][ _HH_MM ]


- **`<animalName>`** — literal animal identifier string from lab notes.  
  - Often begins with an 8-digit surgical date (e.g., `20241015PMCBE1mini`).  
  - That 8-digit prefix is **part of the animal name**, not the experiment day.  
  - May also include things like `v1`, `r1`, `LE1`, `BE0`, or end in `mini`.  
  - These substrings are not parsed unless separated by underscores.
- **`_<tag>`** — optional free-form tokens describing condition.  
  Examples: `social`, `single`, `AO`, `BO`, `paint`, `bleach`, `coated`, `samecage`, etc.
- **`_p<partnerName>`** — optional partner’s literal name (for social sessions).
- **`_HH_MM`** — optional approximate start time (24h).  
  Used when the same session base was recorded multiple times on the same day.

### Examples

- `20240303PMCBE0r1coated`  
  → Animal `20240303PMCBE0r1coated` (surgical date + name).  
  No partner, no time.

- `20240303PMCBE0r1coated_14_53`  
  → Same animal, but recorded again at 14:53 (prevents overwrite).

- `20241015pmcr2_AO_12_52`  
  → Animal `20241015pmcr2`; tag `AO`; time `12:52`.

- `20241212V1RE1L23Fmini_p20241212V1BE1L23F`  
  → Animal `20241212V1RE1L23Fmini`; partner `20241212V1BE1L23F`.

---

## Legacy / alternate patterns

- **Legacy social prefix**:  
  `2social_mini_20240910V1r_AO_single_12_50`  
  → Encodes “two mice / social / miniscope” in the prefix, then adds an animal string + tags + time.  
  Keep for historical data; prefer the canonical pattern for new data.

- **Failure / exclude marker**:  
  A leading `#` marks flagged/failed sessions (e.g., `cam1dead`, `minifailed`).

---

## Calibration folders inside a session

Each session may include calibration subfolders named as:

- `calib_before` → default choice when present.  
- `calib_HH_MM` → calibration done at a specific time (e.g., `calib_14_28`).  
- `calib_after` → calibration done after recording.  

When multiple `calib_*` exist:
1. Prefer `calib_before`.  
2. If not, use the closest `calib_HH_MM`.  
3. Otherwise fall back to `calib_after` or the only available calibration folder.

---

## Tag glossary (extend as needed)

- `social` — social interaction condition  
- `single` — this means the social partner is a specific single housed animal
- `AO` — after oxytocin  
- `BO` — before oxytocin  
- `paint`, `bleach`, `coated` — marking/treatment tags  
- `samecage` — same-cage pairing  
- `mini` (when separated by `_mini`) — miniscope tag  
  - **Note**: if an animal name literally ends with `mini` (e.g. `...BE1mini`), it is indicating which animal has the miniscope.

---

## Disambiguation

- If multiple sessions share the same `<animalName>` on the same `YYYY_MM_DD`, append `_HH_MM` to distinguish.  
- Do not retro-rename historical folders. If you revise a name, keep a mapping table (see `NAME_INDEX.md`).

---

## Machine-readable index

We also maintain an extended CSV/paret with richer fields per session, e.g.:

- `mir_generate_param, sync, com, social, miniscope, test, after_oxytocin, before_oxytocin, dannce, mini_rec_sync, rec_file, scan_time, date_folder, calib_files`

A helper script (`build_name_index.py`) scans sessions, extracts these fields, and writes `NAME_INDEX.md` (auto-generated table).  
That way, newcomers don’t have to guess meanings from folder names alone.



In [ ]:
Oct3V1/
├─ YYYY_MM_DD/                 # experiment day (calendar date of the recording)
│  ├─ <SessionName>/           # a single recording session
│  └─ <SessionName>_HH_MM/     # same session base, different start time (prevents overwrite)
└─ README_Naming.md            # this file


In [ ]:
<SessionName> ::= <animalName> [ "_" <tag> ... ] [ "_p" <partnerName> ] [ "_" <HH> "_" <MM> ]


In [ ]:
<root>/<animal_id>/<custom_label>/<date>/<time>/
└─ My_V4_Miniscope/
   ├─ timeStamps.csv
   └─ *.nc


Dataset Structure & Linkage (Oct3V1 + Miniscope + Siblings)

The filesystem is the API. Paths and fixed filenames define what downstream tools can rely on.
Status fields come from dynamic Python configs (status_fields_config_*.py).

1) Dataset roots and scope

At the top level there are sibling dataset roots that share the same overall folder layout (recording day → session) but may use slightly different naming conventions:

/.../
├─ Oct3V1/             # behavior + 6Cam (canonical naming in this repo)
├─ Oct3V1mini_sorted/  # miniscope data (separate tree)
├─ 24summ/             # same structure, variant naming
└─ <other_roots>/      # additional datasets with the same layout but naming differences


Tools should treat naming tokens as opaque unless a rule is explicitly defined here.

Status detection and optional name parsing are configured in status_fields_config_*.py.

2) Canonical behavior/6Cam layout (Oct3V1)
Oct3V1/
├─ YYYY_MM_DD/                   # experiment day (calendar date of the recording)
│  ├─ <SessionName>/             # a single recording session
│  ├─ <SessionName>_HH_MM/       # same session base, different start time
│  └─ miniscope_pointer.txt      # (date-level) path to miniscope root used that day
└─ README_Naming.md              # naming grammar (kept as your original document)


Some sessions are behavior-only (no miniscope).

miniscope_pointer.txt is a date-level hint only; session-level mapping files override it (see §4).

3) Miniscope layout (separate tree, reciprocal link)
Oct3V1mini_sorted/
└─ <animal_id>/<custom_label>/<date>/<time>/
   ├─ My_V4_Miniscope/
   │  ├─ timeStamps.csv
   │  └─ *.nc
   └─ sync_to_rec_path.txt   # reciprocal pointer back to the Oct3V1 session


Reciprocal mapping: miniscope side provides sync_to_rec_path.txt (sibling of My_V4_Miniscope/).

Behavior/6Cam side can provide sync_to_mini_path.txt inside the session folder (see §4).

4) Cross-dataset linkage policy (Oct3V1 ⇆ Oct3V1mini_sorted)

Resolution order (most specific wins):

Session-level pointer (preferred):

Oct3V1/.../<Session>/sync_to_mini_path.txt → points into Oct3V1mini_sorted/.../<date>/<time>/

Oct3V1mini_sorted/.../<date>/<time>/sync_to_rec_path.txt → points to Oct3V1/.../<Session>/

Date-level pointer (fallback):

Oct3V1/YYYY_MM_DD/miniscope_pointer.txt → miniscope root used that day

No mapping found: behavior-only or unresolved link

Symmetry check:
If both session-level pointers exist (on each side), they should mutually resolve. If they disagree, treat as a conflict and resolve manually.

5) Artifacts inside an Oct3V1 session (behavior/6Cam side)

Within Oct3V1/YYYY_MM_DD/<SessionName>[_HH_MM]/:

Artifact	Relative path	Notes
DANNCE predictions	DANNCE/predict00/save_data_AVG.mat	presence ⇒ dannce = 1
DANNCE visualization	DANNCE/predict00/vis/*.jpg	presence ⇒ dannce_vis = 1
Center of Mass (COM)	COM/predict00/com3d*.mat	presence ⇒ com = 1
COM visualization	COM/predict00/vis/*.jpg	presence ⇒ com_vis = 1
Alignment bundle (behavior+Ca dF/F)	MIR_Aligned/aligned_predictions_with_ca_and_dF_F*.h5	presence ⇒ mini_rec_sync = 1
Alignment bundle (basic)	MIR_Aligned/aligned_predictions_with_ca.h5	presence ⇒ mini_rec_sync = 0.5
Alignment bundle (COM-only)	MIR_Aligned/only_com*.h5	presence ⇒ mini_rec_sync_com = 1
Calibration folders	calib_before/ or calib_HH_MM/ or calib_after/	selection policy in §5.1
Session-level mapping	sync_to_mini_path.txt	overrides date-level pointer
5.1 Calibration selection

calib_before → 2) nearest calib_HH_MM → 3) calib_after → 4) the only calib_* present.

6) Status fields (dynamic; defined in status_fields_config_*.py)

Numeric convention: 0 = NO, 1 = YES, 2 = NO-NEED, 3 = FAILED, and in one case 0.5 = partial.

6A. Pipeline stages (reflect processing steps)

mir_generate_param — YES if a calib_file exists (per your rule).

sync — YES if calib_file basename matches df_*label3d_dannce.mat; FAILED if in failed_paths.

dropf_handle — YES if df_dh_*label3d_dannce.mat; NO-NEED for dates <= 2024_11_01; FAILED if in failed_paths.

com — YES if COM/predict00/com3d*.mat exists.

com_vis — YES if COM/predict00/vis/*.jpg exists.

dannce — YES if DANNCE/predict00/save_data_AVG.mat exists.

dannce_vis — YES if DANNCE/predict00/vis/*.jpg exists.

mini_rec_sync — 1.0 if aligned_predictions_with_ca_and_dF_F*.h5; 0.5 if aligned_predictions_with_ca.h5.

mini_rec_sync_com — YES if only_com*.h5 exists.

Multiple prediction models: this is dynamic. Add/modify detectors in status_fields_config_*.py (e.g., support alternative model folders or glob patterns) without changing this document.

6B. Session properties (inherent attributes)

social — YES if path contains any of: social, mini_p, 2mice, _p, 2male_mice (case-insensitive where applicable).

miniscope — YES if path contains mini (or heuristic tokens like pmc, v1; note this can over-include during tests).

test — YES if path contains test.

after_oxytocin — YES if path contains AO.

before_oxytocin — YES if path contains BO.

mini_6cam_map — YES if session-level sync_to_mini_path.txt exists; FAILED if path is in failed_paths.

This is a linkage property (mapping presence), not a processing stage.

Properties describe the session; stages describe what has been computed.

7) Sibling datasets with variant naming (e.g., 24summ/, others)

Folder structure: same day/session tree; naming tokens may differ.

Artifact contract: keep the same relative paths (e.g., DANNCE/predict00/…, COM/predict00/…, MIR_Aligned/…, My_V4_Miniscope/…).

Mapping files: keep the same filenames and semantics:

On behavior/6Cam side: sync_to_mini_path.txt (session-level), miniscope_pointer.txt (date-level).

On miniscope side: sync_to_rec_path.txt (reciprocal).

Status logic: customize per-root via the appropriate status_fields_config_*.py.

8) Legacy / alternates (kept for completeness)

Legacy prefix form: 2social_mini_20240910V1r_AO_single_12_50 — retained; do not retro-rename.

Failure/exclude marker: a leading # denotes flagged/failed sessions (skip by default).

9) Contributors (placeholder)

R — animal surgery

A — arena setup

J, A, M — assisted with arena setup

M & R — recordings

M — preprocessing + data-management pipeline

T — prediction model training

(Replace initials with full names/affiliations when ready.)


# Dataset Structure & Linkage (Oct3V1 + Miniscope + Siblings)

> The filesystem is the API. Paths and fixed filenames define what downstream tools can rely on.  
> Status fields come from dynamic Python configs (`status_fields_config_*.py`).

---

## 1) Dataset roots and scope

At the top level there are **sibling dataset roots** that share the same overall folder layout (recording day → session), with minor naming differences:

```

/.../
├─ Oct3V1/             # behavior + 6Cam (canonical naming in this repo)
├─ Oct3V1mini\_sorted/  # miniscope data (separate tree)
├─ 24summ/             # same structure, variant naming
└─ \<other\_roots>/      # additional datasets with the same layout but naming differences

```

---

## 2) Canonical behavior/6Cam layout (Oct3V1)

```

Oct3V1/
├─ YYYY\_MM\_DD/                   # experiment day (calendar date of the recording)
│  ├─ <SessionName>/             # a single recording session
│  ├─ <SessionName>\_HH\_MM/       # same session base, different start time
│  └─ miniscope\_pointer.txt      # optional human-readable note indicating miniscope path used that day
└─ README\_Naming.md              # your original naming grammar document

```

- Some sessions are **behavior-only** (no miniscope).
- `miniscope_pointer.txt` may exist for convenience; **authoritative linking** is session-level (see §4).

### 1.1 Session name grammar

```

<SessionName> ::= <animalName> \[ "*" <tag> ... ] \[ "*p" <partnerName> ] \[ "*" <HH> "*" <MM> ]

```

- **`<animalName>`** — literal animal string from lab notes.  
  Often starts with an 8-digit surgical date (e.g., `20241015PMCBE1mini`); this prefix is **part of the name**.  
  Internal tokens like `v1`, `r1`, `LE1`, `BE0`, trailing `mini` are **opaque unless separated by underscores**.
- **`_<tag>`** — free-form descriptors such as `social`, `single`, `AO`, `BO`, `paint`, `bleach`, `coated`, `samecage`, `mini`, …
- **`_p<partnerName>`** — partner’s literal name (social sessions).
- **`_HH_MM`** — optional 24h start time.

**Examples**

- `20240303PMCBE0r1coated`  
- `20240303PMCBE0r1coated_14_53`  
- `20241015pmcr2_AO_12_52`  
- `20241212V1RE1L23Fmini_p20241212V1BE1L23F`

**Disambiguation:** If multiple sessions share the same `<animalName>` on a given day, append `_HH_MM`.

---

## 3) Miniscope layout (separate tree, reciprocal link)

```

Oct3V1mini\_sorted/
└─ \<animal\_id>/\<custom\_label>/<date>/<time>/
├─ My\_V4\_Miniscope/
│  ├─ timeStamps.csv
│  └─ \*.nc
└─ sync\_to\_rec\_path.txt   # reciprocal pointer back to the Oct3V1 session

```

- **Reciprocal mapping:** miniscope side provides `sync_to_rec_path.txt` (sibling of `My_V4_Miniscope/`).

---

## 4) Cross-dataset linkage policy (Oct3V1 ⇆ Oct3V1mini_sorted)

**Authoritative, symmetric pointers:**

- On the **Oct3V1 session side**:  
  `Oct3V1/.../<Session>/sync_to_mini_path.txt` → points into `Oct3V1mini_sorted/.../<date>/<time>/`
- On the **miniscope unit side**:  
  `Oct3V1mini_sorted/.../<date>/<time>/sync_to_rec_path.txt` → points to `Oct3V1/.../<Session>/`

**Symmetry check:** If both pointers exist, they should mutually resolve. If they disagree, treat as a **conflict** and resolve manually.

*(Note: `miniscope_pointer.txt` in the date folder is informational; it is not used for automated linkage.)*

---

## 5) Calibration folder selection

When multiple calibration folders exist:

1. Prefer `calib_before`  
2. Else choose time-nearest `calib_HH_MM`  
3. Else use `calib_after`  
4. Else use the only available `calib_*`

(Record which one you used internally as needed; no index section here.)

---

## 6) Status fields (dynamic; defined in `status_fields_config_*.py`)

**Numeric convention:** `0 = NO`, `1 = YES`, `2 = NO-NEED`, `3 = FAILED`, and in one case `0.5 = partial`.  
Status fields split into **pipeline stages** vs. **session properties**.

### 6A. Pipeline **stages** (processing steps)

- **`mir_generate_param`** — YES if a `calib_file` exists (per your rule).  
- **`sync`** — YES if `calib_file` basename matches `df_*label3d_dannce.mat`; FAILED if in `failed_paths`.  
- **`dropf_handle`** — YES if `df_dh_*label3d_dannce.mat`; NO-NEED for dates `<= 2024_11_01`; FAILED if in `failed_paths`.  
- **`com`** — YES if `COM/predict00/com3d*.mat` exists.  
- **`com_vis`** — YES if `COM/predict00/vis/*.jpg` exists.  
- **`dannce`** — YES if `DANNCE/predict00/save_data_AVG.mat` exists.  
- **`dannce_vis`** — YES if `DANNCE/predict00/vis/*.jpg` exists.  
- **`mini_rec_sync`** — `1.0` if `aligned_predictions_with_ca_and_dF_F*.h5`; `0.5` if `aligned_predictions_with_ca.h5`.  
- **`mini_rec_sync_com`** — YES if `MIR_Aligned/only_com*.h5` exists.

> **Multiple prediction models:** this is **dynamic**—extend the detectors in `status_fields_config_*.py` (e.g., alternate model folders or patterns) without editing this Markdown.

### 6B. Session **properties** (inherent attributes)

- **`social`** — YES if path contains any of: `social`, `mini_p`, `2mice`, `_p`, `2male_mice`.  
- **`miniscope`** — YES if path contains `mini` (or heuristic tokens like `pmc`, `v1`; may over-include for tests).  
- **`test`** — YES if path contains `test`.  
- **`after_oxytocin`** — YES if path contains `AO`.  
- **`before_oxytocin`** — YES if path contains `BO`.  
- **`mini_6cam_map`** — YES if **session-level** `sync_to_mini_path.txt` exists; FAILED if path is in `failed_paths`.  
  - This is a **linkage property** (mapping presence), not a processing stage.

---

## 7) Sibling datasets with variant naming (e.g., `24summ/`, others)

- **Folder structure:** same day/session tree; naming tokens may differ.  
- **Artifact contract:** keep the **same relative paths** (e.g., `DANNCE/predict00/…`, `COM/predict00/…`, `MIR_Aligned/…`, `My_V4_Miniscope/…`).  
- **Mapping files:** keep the same filenames and semantics:  
  - On behavior/6Cam side: `sync_to_mini_path.txt` (session-level).  
  - On miniscope side: `sync_to_rec_path.txt` (reciprocal).  
- **Status logic:** customize per-root via the appropriate `status_fields_config_*.py`.

---

## 8) Legacy / alternates

- **Legacy prefix form:** `2social_mini_20240910V1r_AO_single_12_50` — retained; do not retro-rename.  
- **Failure/exclude marker:** a leading `#` denotes flagged/failed sessions (skip by default).

---

## 9) Contributors (placeholder)

- Renzhi Zhan — animal surgery  
- Anshuman Sabath — arena setup  
- Lingxuan (Mir) Qi, Junyu Nan, Annelise Brigham — assisted with arena setup  
- Renzhi Zhan & Lingxuan (Mir) Qi — recordings  
- Lingxuan (Mir) Qi — preprocessing + data-management pipeline, BBOP  
- Tianqing Li — prediction model training & database consulting



# Dataset Structure & Linkage (Oct3V1 + Miniscope + Siblings)

> The filesystem is the API. Paths and fixed filenames define what downstream tools can rely on.  
> Status fields come from dynamic Python configs (`status_fields_config_*.py`).

---

## 1) Dataset roots and scope

At the top level there are **sibling dataset roots** that share the same overall folder layout (recording day → session), with minor naming differences:

```

/.../
├─ Oct3V1/             # behavior + 6Cam (canonical naming in this repo)
├─ Oct3V1mini\_sorted/  # miniscope data (separate tree)
├─ 24summ/             # same structure, variant naming
└─ \<other\_roots>/      # additional datasets with the same layout but naming differences

```

### 1.1 Session name grammar

```

<SessionName> ::= <animalName> \[ "*" <tag> ... ] \[ "*p" <partnerName> ] \[ "*" <HH> "*" <MM> ]

```

- **`<animalName>`** — literal animal string from lab notes.  
  Often starts with an 8-digit surgical date (e.g., `20241015PMCBE1mini`); this prefix is **part of the name**.  
  Internal tokens like `v1`, `r1`, `LE1`, `BE0`, trailing `mini` are **opaque unless separated by underscores**.
- **`_<tag>`** — free-form descriptors such as `social`, `single`, `AO`, `BO`, `paint`, `bleach`, `coated`, `samecage`, `mini`, …
- **`_p<partnerName>`** — partner’s literal name (social sessions).
- **`_HH_MM`** — optional 24h start time.

**Examples**

- `20240303PMCBE0r1coated`  
- `20240303PMCBE0r1coated_14_53`  
- `20241015pmcr2_AO_12_52`  
- `20241212V1RE1L23Fmini_p20241212V1BE1L23F`

**Disambiguation:** If multiple sessions share the same `<animalName>` on a given day, append `_HH_MM`.

---

## 2) Canonical behavior/6Cam layout (Oct3V1)

```

Oct3V1/
├─ YYYY\_MM\_DD/                   # experiment day (calendar date of the recording)
│  ├─ <SessionName>/             # a single recording session
│  ├─ <SessionName>\_HH\_MM/       # same session base, different start time
│  └─ miniscope\_pointer.txt      # optional human-readable note indicating miniscope path used that day
└─ README\_Naming.md              # your original naming grammar document

```

- Some sessions are **behavior-only** (no miniscope).
- `miniscope_pointer.txt` may exist for convenience; **authoritative linking** is session-level (see §4).

---

## 3) Miniscope layout (separate tree, reciprocal link)

```

Oct3V1mini\_sorted/
└─ \<animal\_id>/\<custom\_label>/<date>/<time>/
├─ My\_V4\_Miniscope/
│  ├─ timeStamps.csv
│  └─ \*.nc
└─ sync\_to\_rec\_path.txt   # reciprocal pointer back to the Oct3V1 session

```

- **Reciprocal mapping:** miniscope side provides `sync_to_rec_path.txt` (sibling of `My_V4_Miniscope/`).

---

## 4) Cross-dataset linkage policy (Oct3V1 ⇆ Oct3V1mini_sorted)

**Authoritative, symmetric pointers:**

- On the **Oct3V1 session side**:  
  `Oct3V1/.../<Session>/sync_to_mini_path.txt` → points into `Oct3V1mini_sorted/.../<date>/<time>/`
- On the **miniscope unit side**:  
  `Oct3V1mini_sorted/.../<date>/<time>/sync_to_rec_path.txt` → points to `Oct3V1/.../<Session>/`

**Symmetry check:** If both pointers exist, they should mutually resolve. If they disagree, treat as a **conflict** and resolve manually.

*(Note: `miniscope_pointer.txt` in the date folder is informational; it is not used for automated linkage.)*

---



## 5) Status fields (dynamic; defined in `status_fields_config_*.py`)

**Numeric convention:** `0 = NO`, `1 = YES`, `2 = NO-NEED`, `3 = FAILED`, and in one case `0.5 = partial`.  
Status fields split into **pipeline stages** vs. **session properties**.

### 5.1. Pipeline **stages** (processing steps)

- **`mir_generate_param`** — YES if a `calib_file` exists (per your rule).  
- **`sync`** — YES if `calib_file` basename matches `df_*label3d_dannce.mat`; FAILED if in `failed_paths`.  
- **`dropf_handle`** — YES if `df_dh_*label3d_dannce.mat`; NO-NEED for dates `<= 2024_11_01`; FAILED if in `failed_paths`.  
- **`com`** — YES if `COM/predict00/com3d*.mat` exists.  
- **`com_vis`** — YES if `COM/predict00/vis/*.jpg` exists.  
- **`dannce`** — YES if `DANNCE/predict00/save_data_AVG.mat` exists.  
- **`dannce_vis`** — YES if `DANNCE/predict00/vis/*.jpg` exists.  
- **`mini_rec_sync`** — `1.0` if `aligned_predictions_with_ca_and_dF_F*.h5`; `0.5` if `aligned_predictions_with_ca.h5`.  
- **`mini_rec_sync_com`** — YES if `MIR_Aligned/only_com*.h5` exists.

> **Multiple prediction models:** this is **dynamic**—extend the detectors in `status_fields_config_*.py` (e.g., alternate model folders or patterns) without editing this Markdown.

### 5.2. Session **properties** (inherent attributes)

- **`social`** — YES if path contains any of: `social`, `mini_p`, `2mice`, `_p`, `2male_mice`.  
- **`miniscope`** — YES if path contains `mini` (or heuristic tokens like `pmc`, `v1`; may over-include for tests).  
- **`test`** — YES if path contains `test`.  
- **`after_oxytocin`** — YES if path contains `AO`.  
- **`before_oxytocin`** — YES if path contains `BO`.  
- **`mini_6cam_map`** — YES if **session-level** `sync_to_mini_path.txt` exists; FAILED if path is in `failed_paths`.  
  - This is a **linkage property** (mapping presence), not a processing stage.

---

## 6) Sibling datasets with variant naming (e.g., `24summ/`, others)

- **Folder structure:** same day/session tree; naming tokens may differ.  
- **Artifact contract:** keep the **same relative paths** (e.g., `DANNCE/predict00/…`, `COM/predict00/…`, `MIR_Aligned/…`, `My_V4_Miniscope/…`).  
- **Mapping files:** keep the same filenames and semantics:  
  - On behavior/6Cam side: `sync_to_mini_path.txt` (session-level).  
  - On miniscope side: `sync_to_rec_path.txt` (reciprocal).  
- **Status logic:** customize per-root via the appropriate `status_fields_config_*.py`.

---
## 7) Calibration folder selection

When multiple calibration folders exist:

1. Prefer `calib_before`  
2. Else choose time-nearest `calib_HH_MM`  
3. Else use `calib_after`  
4. Else use the only available `calib_*`

(Record which one you used internally as needed; no index section here.)

---
## 8) Legacy / alternates

- **Legacy prefix form:** `2social_mini_20240910V1r_AO_single_12_50` — retained; do not retro-rename.  
- **Failure/exclude marker:** a leading `#` denotes flagged/failed sessions (skip by default).

---

## 9) Contributors (placeholder)

- Renzhi Zhan — animal surgery  
- Anshuman Sabath — arena setup  
- Lingxuan (Mir) Qi, Junyu Nan, Annelise Brigham — assisted with arena setup  
- Renzhi Zhan & Lingxuan (Mir) Qi — recordings  
- Lingxuan (Mir) Qi — preprocessing + data-management pipeline, BBOP  
- Tianqing Li — prediction model training & database consulting



# Processing Pipeline (Chronological Execution)

**Scope.** End-to-end processing from raw acquisition → synchronized, integrated outputs.  
**Data roots.** `Oct3V1/` (behavior + 6Cam), `Oct3V1mini_sorted/` (miniscope), sibling sets like `24summ/` share the same layout.

---

## 0) Pointers & session linkage (authoritative, symmetric)

- **Oct3V1 session → miniscope:** `sync_to_mini_path.txt` (inside the session folder)
- **Miniscope unit → Oct3V1 session:** `sync_to_rec_path.txt` (sibling of `My_V4_Miniscope/`)
- If both exist they should agree; if not, resolve manually.

---

## 1) High-level flow

```

Acquisition (Miniscope + SixCam)
├─► Raw Miniscope (My\_V4\_Miniscope/)
│
└─► Raw SixCam (6 files, same session ts; frame starts differ)
└─► SixCam Internal Sync (light-drop offsets)

Miniscope Ca2+ Extraction                       SixCam Keypoints/CoM (DANNCE)
(motion correction, ROI, ΔF/F, QC)              (predictions + validation)

```
          └──────────── Miniscope ↔ SixCam Sync (post-preprocessing) ────────────┘
                               (light-drop + offsets)

                      └─► Integration Outputs (MIR_Aligned/)
                      │      - frame_mapping.json
                      │      - aligned CoM
                      │      - aligned keypoints (3D)
                      │      - raw + 20% ΔF/F
                      └─► Downstream Analysis (Ca traces, ROI alignment, CoM/keypoints, cross-modal)
```

```

---

## 2) Stage table (inputs → scripts/notebooks → outputs → status fields)

| Stage | Inputs | Script / Notebook | Outputs (relative to session) | Status fields touched |
|------:|-------|--------------------|-------------------------------|-----------------------|
| A | Raw 6Cam files | **`scan_20250625`** (local) | internal-sync offsets via light drop | `sync` (via calib_file rule) |
| B | A + calib | **`ssh_exe_oct3v1*.ipynb`** (cluster) | `DANNCE/predict00/save_data_AVG.mat`; `COM/predict00/com3d*.mat` | `dannce`, `com` |
| C | B | validator steps in the same notebooks | `DANNCE/predict00/vis/*.jpg`, `COM/predict00/vis/*.jpg` | `dannce_vis`, `com_vis` |
| D | Miniscope raw | **`miniparam_2`** (local) | `My_V4_Miniscope/timeStamps.csv`, `*.nc` processed; ΔF/F; QC figs | (none; properties like `miniscope` are separate) |
| E | B + D + mapping files | **bundle-alignment script** | `MIR_Aligned/aligned_predictions_with_ca_and_dF_F*.h5` (or `aligned_predictions_with_ca.h5`, `only_com*.h5`); `frame_mapping.json` | `mini_rec_sync` (1.0/0.5), `mini_rec_sync_com` |
| F | E | loaders (flexible to new preds) | pandas DataFrames for analysis | — |

> Loaders consume `frame_mapping.json` so **new predictions** can be dropped in and re-loaded without re-running the whole pipeline.

---

## 3) Calibration folder selection (used where needed)

When multiple calibration folders exist in a session:

1. Prefer `calib_before`  
2. Else choose time-nearest `calib_HH_MM`  
3. Else use `calib_after`  
4. Else use the only `calib_*` present

(Record which one was used internally; no index file required.)

---

## 4) Scripts & notebooks (what they do)

- **`scan_20250625` (local)** — discovers sessions; triggers 6Cam pre-DANNCE steps; prepares paths/artifacts for cluster runs; runs post-prediction validation when results return.
- **`ssh_exe_oct3v1*.ipynb` (cluster)** — runs DANNCE keypoint prediction and CoM prediction; saves `save_data_AVG.mat`, `com3d*.mat`, and optional vis.
- **`miniparam_2` (local)** — miniscope pipeline (motion correction, ROI, ΔF/F, QC). Some manual steps remain; script loops targets and visualizes results.
- **Bundle alignment script** — fuses Ca and predictions using light-drop & offsets; emits `MIR_Aligned/*` and `frame_mapping.json`.
- **Loaders** — read `frame_mapping.json` and aligned bundles; allow swapping in **new predictions** without repeating extraction.

---

## 5) Status fields (summary; defined in `status_fields_config_*.py`)

**Pipeline stages:** `mir_generate_param`, `sync`, `dropf_handle`, `dannce`, `dannce_vis`, `com`, `com_vis`, `mini_rec_sync` (1.0/0.5), `mini_rec_sync_com`.  
**Properties:** `social`, `miniscope`, `test`, `after_oxytocin`, `before_oxytocin`, `mini_6cam_map`.

- Multiple model variants are supported by editing the detectors in `status_fields_config_*.py` (e.g., alternate model folders or glob patterns).

---

## 6) Re-runs & idempotence

- Stages write deterministic artifacts under fixed relative paths.  
- Re-running a stage should **overwrite only its own outputs**; downstream stages re-use `frame_mapping.json`.  
- Failure handling: mark via `failed_paths` list; sessions prefixed with `#` are ignored by default.

---

## 7) Analysis entry points

- Use loaders to obtain a single `DataFrame` that includes: Ca traces/ΔF/F, ROI metadata, aligned 3D keypoints, CoM, and mapping indices for join operations.
```


<!-- 
---

### `RUNBOOK.md` (execution playbook)

```markdown -->
# Execution Playbook (Local + Cluster)

This is the practical “how to run it” guide. It complements `PROCESSING.md`.

---

## 0) Pre-reqs

- Data roots available: `Oct3V1/`, `Oct3V1mini_sorted/` (plus siblings like `24summ/`)
- Session linkage files: `sync_to_mini_path.txt` and/or `sync_to_rec_path.txt`
- Envs: local (miniscope + pre-/post-processing), cluster (DANNCE prediction)

---

## 1) Local: discover and prep

1. Run **`scan_20250625`** against target day(s) or sessions.  
   - Produces/updates paths expected by cluster notebooks.  
   - Verifies presence of mapping files and calibration folder per policy.

2. (Optional) If linkage missing, add:  
   - `Oct3V1/.../<Session>/sync_to_mini_path.txt`  
   - `Oct3V1mini_sorted/.../<date>/<time>/sync_to_rec_path.txt`

---

## 2) Cluster: predictions

1. Launch **`ssh_exe_oct3v1*.ipynb`** on the cluster with the prepared session list.  
2. Outputs per session:  
   - `DANNCE/predict00/save_data_AVG.mat`  
   - `COM/predict00/com3d*.mat`  
   - `*/vis/*.jpg` (optional)

> If using alternate models, set paths according to your `status_fields_config_*.py` variant.

---

## 3) Local: miniscope extraction

- Run **`miniparam_2`** over the miniscope units linked to each session.  
- Outputs live under `My_V4_Miniscope/` (timestamps, `*.nc`, ΔF/F, QC figs).

---

## 4) Local: bundle alignment

- Run the **bundle alignment script** for each session once predictions and miniscope outputs are ready.  
- Emits under `MIR_Aligned/`:  
  - `aligned_predictions_with_ca_and_dF_F*.h5` (full)  
  - or `aligned_predictions_with_ca.h5` (basic)  
  - or `only_com*.h5` (COM-only)  
  - plus `frame_mapping.json`

---

## 5) Validation & loaders

- Use validator helpers to check alignment sanity (light-drop peaks, offset consistency, frame coverage).  
- Use **loaders** to assemble pandas DataFrames for analysis.  
- To **swap in new predictions**, drop them in place and re-run loaders (no need to repeat Ca extraction or alignment if mapping is unchanged).

---

## 6) Re-run policy

- Re-run a stage only if its inputs changed or validation failed.  
- Use `failed_paths` to skip problematic sessions until fixed.  
- Keep calibration choice consistent with the policy in `PROCESSING.md §3`.

---
